# Hour 2 · Keywords and TF-IDF


**Goal:** automatically surface the vocabulary *characteristic* of each tablet and genre.

**Needs:** `pip install scikit-learn pandas` (in `requirements.txt`).

*Reading:* `docs/04-genres.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first ===
# Loads the Copenhagen Ugaritic Corpus (CUC) from the HuggingFace JSONL cache. No edits needed.
import sys
sys.path.append("..")                       # so Python can find data/loader.py
from data.loader import load_texts

texts = load_texts()                        # 278 real KTU tablets; cached after first download
print(f"Loaded {len(texts)} tablets.")
texts[0]["ktu"], texts[0]["genre"], texts[0]["lines"][:2]

## 1. Pick a genre-diverse working set
The full corpus is big; start with a readable mix.

In [ ]:
sample = [t for t in texts if t["genre"] in {"myth","letter","ritual","divination"}
          and len(t["tokens"]) >= 25]
print(len(sample), "tablets")
from collections import Counter
print(Counter(t["genre"] for t in sample))

## 2. Plain word frequencies first

In [ ]:
from data.loader import token_counts
token_counts(sample).most_common(15)

## 3. TF-IDF — distinctive words per tablet
TF-IDF rewards words frequent *in one tablet* but rare *across the corpus*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from data.loader import corpus_as_documents
labels, docs = corpus_as_documents(sample)
vec = TfidfVectorizer(token_pattern=r"[^\s]+")
X = vec.fit_transform(docs)
print("matrix (tablets x vocab):", X.shape)

## 4. Top keywords per tablet — can you guess the genre?

In [ ]:
import numpy as np
feat = np.array(vec.get_feature_names_out())
def top_kw(i, n=6):
    row = X[i].toarray().ravel()
    idx = row.argsort()[::-1][:n]
    return [feat[j] for j in idx if row[j] > 0]
for i, t in enumerate(sample):
    print(f'{t["ktu"]:7s} [{t["genre"]:11s}] {top_kw(i)}')

## 5. Discussion
- How much of the genre is visible from the keywords alone?
- **Caveats:** word *forms* vs lemmas (CUC 0.1.x has no lemmatisation — homographs blur the signal); short/damaged tablets; genre formulas dominating.

> **Production version:** UgaritLab (`dulat/scripts/generate_stats.py`) computes TF-IDF over the whole corpus and projects it for browsing.